# 02 - Download PubMed Abstracts

Fetches full PubMed abstracts from the NCBI E-utilities API for all 1,000 PubMedQA samples.
These raw abstracts serve as the document corpus for vector DB ingestion.

**Input:** PubMedQA dataset (from notebook 01)  
**Output:** `data/raw/{pubid}.txt` files (one per abstract)

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from src.data_loader import load_pubmedqa
from src.abstract_fetcher import fetch_pubmed_abstract, download_all_abstracts, fetch_abstracts_for_mesh
import config

## Load Dataset

In [ ]:
df = load_pubmedqa()
print(f"Total samples to download: {len(df)}")

## Test Single Fetch

In [ ]:
sample_pubid = df.iloc[0]["pubid"]
sample_abstract = fetch_pubmed_abstract(sample_pubid)
print(f"PubID: {sample_pubid}")
print(f"Abstract length: {len(sample_abstract)} chars")
print(f"Preview:\n{sample_abstract[:500]}")

## Download All Abstracts

Downloads all 1,000 abstracts with a 1-second delay between requests to respect NCBI rate limits.  
Already-downloaded files are skipped automatically.

In [ ]:
download_all_abstracts(df, output_dir=config.DATA_RAW_DIR, delay=1.0)

## Verify Downloads

In [ ]:
import os

downloaded = [f for f in os.listdir(config.DATA_RAW_DIR) if f.endswith(".txt")]
print(f"Downloaded {len(downloaded)} abstracts out of {len(df)} expected")

# Check for any missing
downloaded_ids = {f.replace('.txt', '') for f in downloaded}
expected_ids = {str(pid) for pid in df['pubid']}
missing = expected_ids - downloaded_ids
if missing:
    print(f"Missing {len(missing)} abstracts: {list(missing)[:10]}...")
else:
    print("All abstracts downloaded successfully.")